## Creating Evaluation Data set as golden in confident AIb

In [1]:
test_data = [
{
    "input":"Who is the current president of USA",
    "expected_output":"Donald Trump"
},
{
    "input":"Who created GPT Model",
    "expected_output":"Open AI"
}
]

In [2]:
from deepeval.dataset import Golden


new_golden = Golden(input="abc")
new_golden.expected_output="hello"

print(new_golden)

input='abc' actual_output=None expected_output='hello' context=None retrieval_context=None additional_metadata=None comments=None tools_called=None expected_tools=None source_file=None name=None custom_column_key_values=None multimodal=False images_mapping=None


In [3]:
from deepeval.dataset import Golden,EvaluationDataset

goldens = []

for dataSet in test_data:
    golden = Golden(
        input=dataSet["input"],
        expected_output=dataSet["expected_output"]
    ) 
    
    goldens.append(golden)

new_dataSet=EvaluationDataset(goldens=goldens)   
new_dataSet

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of USA', actual_output=None, expected_output='Donald Trump', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input='Who created GPT Model', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None)], _alias=None, _id=None, _multi_turn=False)

## push to confident AI

In [4]:
import os
DEEP_EVAL_API_KEY = os.getenv("DEEV_EVAL_KEY")
import deepeval
deepeval.login(DEEP_EVAL_API_KEY)
# new_dataSet.push(alias="new_dataSet_someCompany")

🎉🥳 Congratulations! You've successfully logged in! 🙌

## pull data set from confident AI

In [10]:

cloud_data_set = EvaluationDataset()
cloud_data_set.pull(alias="new_dataSet_someCompany")
cloud_data_set

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of USA', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input='Who created GPT Model', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input='who invented electricity?\n', actual_output=None, expected_output="The invention of electricity is attributed to many scientists and inventors over centuries, but one of the most famous figures associated with the discovery of electricity is Benjamin Franklin. Franklin is often considered the father of electricity due 

In [12]:
cloud_data_set.goldens

[Golden(input='Who is the current president of USA', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None),
 Golden(input='Who created GPT Model', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None),
 Golden(input='who invented electricity?\n', actual_output=None, expected_output="The invention of electricity is attributed to many scientists and inventors over centuries, but one of the most famous figures associated with the discovery of electricity is Benjamin Franklin. Franklin is often considered the father of electricity due to his pioneering work in the field. Ho

 create a list of LLMTestCases from cloud data set

In [17]:
import sys

from deepeval.test_case import LLMTestCase
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# 2. Add that root folder to the path
if project_root not in sys.path:
    sys.path.insert(0, project_root) # insert at 0 to prioritize this path
from modules.POST_LLMCPP_Call import get_llm_response
from modules.local_model_llama import local_judge

test_cases = []
for data in cloud_data_set.goldens:
    test_cases.append( LLMTestCase(
        input=data.input,
        actual_output=get_llm_response(data.input),
        expected_output=data.expected_output
    ))

The current president of the United States is Joe Biden. He took office on January 20, 2021.
The Generative Pre-trained Transformer (GPT) model was created by three research teams from the Facebook AI Research (FAIR) lab in Menlo Park, California. The three teams were:

1. David Luan, Fei-Fei Li, and Alan Yuille from the University of California, Berkeley
2. Ali Farhadi, Ali Ramezani, and Chris Williams from the University of Cambridge
3. Greg Corrado and Mike Schuster from the Google Research team

The GPT model was first publicly announced and released on March 13, 2020, by the Facebook AI Research (FAIR) lab. It was released as a research project to demonstrate the potential of language modeling techniques on large datasets.
There have been several inventors who have contributed to the discovery and development of electricity, but it is not accurate to say that any one person invented electricity. The discovery of electricity dates back to ancient times, with early civilizations usi

In [18]:
test_cases

[LLMTestCase(input='Who is the current president of USA', actual_output='The current president of the United States is Joe Biden. He took office on January 20, 2021.', expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='Who created GPT Model', actual_output='The Generative Pre-trained Transformer (GPT) model was created by three research teams from the Facebook AI Research (FAIR) lab in Menlo Park, California. The three teams were:\n\n1. David Luan, Fei-Fei Li, and Alan Yuille from the University of California, Berkeley\n2. Ali Farhadi, Ali Ramezani, and Chris Williams from the University of Cambridge\n3. Greg Corrado and Mike Schuster from the Google Research team\n\nThe

In [22]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(threshold=0.7,model=local_judge)

evaluate(test_cases=test_cases,metrics=[metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using Llama-3.2-3B-Instruct-Q4_K_M, strict=False, 
async_mode=True)...

d:\pyhton_projects\Test_AI_LLM_Apps\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: Llama-3.2-3B-Instruct-Q4_K_M, reason: The score is 1.00 because there are no irrelevant statements in the actual output., error: None)

For test case:

  - input: Who is the current president of USA
  - actual output: The current president of the United States is Joe Biden. He took office on January 20, 2021.
  - expected output: Joe Biden
  - context: None
  - retrieval context: None


Metrics Summary

  - ❌ Answer Relevancy (score: 0.6666666666666666, threshold: 0.7, strict: False, evaluation model: Llama-3.2-3B-Instruct-Q4_K_M, reason: The score is 0.67 because all statements are directly related to the invention of electricity., error: None)

For test case:

  - input: who invented electricity?

  - actual output: There have been several inventors who have contributed to the discovery and development of electricity, but it is not accurate to say that any one person invented elect

⚠ WARNING: No hyperparameters logged.
» ]8;id=219073;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=947480;https://app.confident-ai.com/project/cmnk78c9s00daqb16t6rm9b3w/test-runs/cmno2j6t4003fod16dxgz7n9s/regression-testing\https://app.confident-ai.com/project/cmnk78c9s00daqb16t6rm9b3w/test-runs/cmno2j6t4003fod16dxgz7n9s/regression-testi]8;;\
]8;id=947480;https://app.confident-ai.com/project/cmnk78c9s00daqb16t6rm9b3w/test-runs/cmno2j6t4003fod16dxgz7n9s/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because there are no irrelevant statements in the actual output.', strict_mode=False, evaluation_model='Llama-3.2-3B-Instruct-Q4_K_M', error=None, evaluation_cost=None, verbose_logs='Statements:\n[\n    "The current president of the United States is Joe Biden.",\n    "He took office on January 20, 2021."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "yes",\n        "reason": "The statements directly answer the question about the current president of the USA."\n    }\n]')], conversational=False, multimodal=False, input='Who is the current president of USA', actual_output='The current president of the United States is Joe Biden. He took office on January 20, 2021.', expected_output='Joe Biden', context=None, retrieval_context=None, 

In [25]:
test_cases

[LLMTestCase(input='Who is the current president of USA', actual_output='The current president of the United States is Joe Biden. He took office on January 20, 2021.', expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='Who created GPT Model', actual_output='The Generative Pre-trained Transformer (GPT) model was created by three research teams from the Facebook AI Research (FAIR) lab in Menlo Park, California. The three teams were:\n\n1. David Luan, Fei-Fei Li, and Alan Yuille from the University of California, Berkeley\n2. Ali Farhadi, Ali Ramezani, and Chris Williams from the University of Cambridge\n3. Greg Corrado and Mike Schuster from the Google Research team\n\nThe

In [29]:
results = []
for test_case in test_cases:
    score = metric.measure(test_case)
    results.append(score)
# metric.measure(test_cases)
results

d:\pyhton_projects\Test_AI_LLM_Apps\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[Confident AI Metric Data Log] Successfully posted metric data (0 metrics remaining in queue, 1 in flight) 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an environment variable.

[Confident AI Metric Data Log] Successfully posted metric data (0 metrics remaining in queue, 1 in flight) 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an environment variable.

[1.0, 0.6666666666666666, 1.0]

[Confident AI Metric Data Log] Successfully posted metric data (0 metrics 
remaining in queue, 1 in flight) 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an 
environment variable.
